In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

In [2]:
df = pd.read_csv('train.csv')

In [15]:
print(df.isnull().sum()) #Пустые строки в столбцах
print()
print(df["Survived"].value_counts()) #Количество выживших/невыживших
print()
print(df["Survived"].value_counts(normalize=True)) #доли

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Survived
0    549
1    342
Name: count, dtype: int64

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


Предобработка и разбивка на тестовую и тренировочную выборки


In [4]:
median_age = df["Age"].median()
df["Age"] = df["Age"].fillna(median_age)
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

col = ["Sex", "Age"]
X = df[col]
y = df["Survived"]

rd_state = 40
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=rd_state, test_size=0.20, stratify=y)

In [7]:
def print_metrics(model_name, y_true, y_pred, y_prob=None):
    print(f"\n{'='*55}")
    print(f"{model_name}")
    print(f"{'='*55}")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 score: {f1_score(y_true, y_pred, zero_division=0):.4f}")
    if y_prob is not None:
        print(f"ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}")
    print()
    print(classification_report(y_true, y_pred,target_names=["Не выжил", "Выжил"],zero_division=0))

In [13]:
baseline = DummyClassifier(strategy="most_frequent", random_state=rd_state)
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)
print_metrics("БЕЙСЛАЙН",y_test, y_pred_baseline)


БЕЙСЛАЙН
Accuracy: 0.6145
F1 score: 0.0000

              precision    recall  f1-score   support

    Не выжил       0.61      1.00      0.76       110
       Выжил       0.00      0.00      0.00        69

    accuracy                           0.61       179
   macro avg       0.31      0.50      0.38       179
weighted avg       0.38      0.61      0.47       179



Logistic regression

In [19]:
lr = LogisticRegression(max_iter=1000,random_state=rd_state)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1] #вероятность класса 1(выжил)

print_metrics("logistic regression", y_test, y_pred_lr, y_prob_lr)

coef = pd.DataFrame({"Признак": col,"Коэффициент": lr.coef_[0].round(3)}).sort_values("Коэффициент", ascending=False)

print("Коэффициенты модели:")
print(coef.to_string(index=False))


logistic regression
Accuracy: 0.8156
F1 score: 0.7519
ROC-AUC: 0.7870

              precision    recall  f1-score   support

    Не выжил       0.83      0.87      0.85       110
       Выжил       0.78      0.72      0.75        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.82      0.81       179

Коэффициенты модели:
Признак  Коэффициент
    Sex        2.331
    Age       -0.009


У признака Sex большой положительный коэффициент. Следовательно, женщины выживали чаще, чем мужчины. Пол является основным фактором влияющим на выживаемость,т.к признак Age оторицательный.

In [27]:
print("\n" + "="*55)
print("ИТОГ:")
print("="*55)
print(f"  {'Модель':<28} {'F1':>6}  {'ROC-AUC':>8}")
print("-"*55)

results = [("Бейслайн",y_pred_baseline, None),("Logistic Regression", y_pred_lr,y_prob_lr),]

for name, y_pred, y_prob in results:
    f1  = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob) if y_prob is not None else float("nan")
    print(f"  {name:<28} {f1:>6.4f}  {auc:>8.4f}")


ИТОГ:
  Модель                           F1   ROC-AUC
-------------------------------------------------------
  Бейслайн                     0.0000       nan
  Logistic Regression          0.7519    0.7870


в дата-сете выживших 38%, не выживших 61%. получается сильный дисбаланс.
так как accuracy чувствительна к дисбалансу она не подошла. выбраны f1 и roc-auc, потому что первая хорошо работает с дисбалансом, вторая показывает насколько хорошо модель разделяет классы.
Logistic Regression F1 =0.7519, ROC-AUC =0.7870 что значительно лучше бейслайна. Модель научилась находить выживших, а не угадывать их.